# Rove La Mer Beach — Review Analysis & Forecasting

Guest review analysis (6,392 Booking.com reviews, 2023-08-17 to 2026-08-16) covering stay-duration patterns, guest-type breakdowns, and NeuralProphet-based review-score forecasting.

## Setup

In [ ]:
# torch is pinned <2.6 because PyTorch 2.6 changed torch.load's default
# weights_only to True, which breaks NeuralProphet's internal learning-rate
# finder checkpointing with an UnpicklingError. Pinning avoids that entirely.
!pip install neuralprophet "torch<2.6" plotly-resampler

In [ ]:
import pandas as pd
main_df = pd.read_json("rove_la_mer_reviews_full_v2.jsonl", lines=True)

## Data Overview

In [ ]:
main_df.head()

In [ ]:
main_df.info()

In [ ]:
print("Date range:", main_df['reviewed_date'].min(), "to", main_df['reviewed_date'].max())
print("Columns:", list(main_df.columns))

## Feature Engineering

In [ ]:
main_df[["checkin_date", "checkout_date"]] = main_df[["checkin_date", "checkout_date"]].apply(pd.to_datetime)

In [ ]:
main_df["stay_duration"] = (main_df["checkout_date"] - main_df["checkin_date"]).dt.days

## Stay Duration Analysis

In [ ]:
total_days = main_df['stay_duration'].sum()
print(f"Total cumulative nights stayed across all bookings: {total_days:,}")

In [ ]:
x = main_df.groupby('checkin_date')['stay_duration'].sum().reset_index(name='total_stay_duration').sort_values('total_stay_duration', ascending=False)
y = main_df["stay_duration"].max()

print(x.head(10))
print("\nMaximum stay duration in days:", y, "days")
print(main_df[main_df["stay_duration"] == y][["checkin_date", "checkout_date", "stay_duration"]])

## Guest Type Breakdown

In [ ]:
df_by_guest_type = main_df.groupby(['reviewed_date', 'guest_type'])['review_score'].agg(['mean', 'sum']).reset_index()
print(df_by_guest_type)

df_by_guest_type_summary = main_df.groupby('guest_type')['review_score'].agg(['mean', 'sum', 'count']).reset_index()
print(df_by_guest_type_summary)

## Time Series Forecasting — Overall Trend

In [ ]:
daily_df = main_df.groupby('reviewed_date')['review_score'].mean().reset_index()
daily_df.columns = ['ds', 'y']
daily_df['ds'] = pd.to_datetime(daily_df['ds'])

from neuralprophet import NeuralProphet
m = NeuralProphet()
# learning_rate is set explicitly to skip NeuralProphet's automatic
# learning-rate finder -- that step saves/reloads a checkpoint internally,
# which is what triggers the UnpicklingError under PyTorch 2.6+ (see Setup).
metrics = m.fit(daily_df, freq='D', learning_rate=0.1)

future = m.make_future_dataframe(daily_df, periods=30)
forecast = m.predict(future)
forecast['yhat1'] = forecast['yhat1'].clip(1, 10)  # review scores can't exceed 10

In [ ]:
import matplotlib.pyplot as plt
import plotly

fig_forecast = m.plot(forecast)
fig_forecast.show()

fig_components = m.plot_components(forecast)
fig_components.show()

fig_model = m.plot_parameters()
fig_model.show()

## Time Series Forecasting — By Guest Type

In [ ]:
from neuralprophet import NeuralProphet
import pandas as pd

results = {}
forecasts = []

for guest_type in main_df['guest_type'].dropna().unique():
    subset = main_df[main_df['guest_type'] == guest_type]

    daily = subset.groupby('reviewed_date')['review_score'].mean().reset_index()
    daily.columns = ['ds', 'y']
    daily['ds'] = pd.to_datetime(daily['ds'])

    print(f"\n=== {guest_type}: {len(subset)} reviews, {len(daily)} days with data ===")
    if len(daily) < 16:
        print(f"  Skipping -- too little data ({len(daily)} days) for a meaningful fit.")
        continue

    m = NeuralProphet()
    m.fit(daily, freq='D', learning_rate=0.1)

    future = m.make_future_dataframe(daily, periods=10)
    forecast = m.predict(future)
    forecast['guest_type'] = guest_type

    results[guest_type] = m
    forecasts.append(forecast[['ds', 'yhat1', 'guest_type']])

all_forecasts = pd.concat(forecasts, ignore_index=True)
all_forecasts['yhat1'] = all_forecasts['yhat1'].clip(1, 10)

In [ ]:
import plotly.express as px

fig = px.line(all_forecasts, x='ds', y='yhat1', color='guest_type',
              title='Predicted review score, next 10 days, by guest type')
fig.add_hline(y=10, line_dash="dot", line_color="gray")  # visual reminder of the real ceiling
fig.show()

## Model Validation (Backtest)

Train on all but the most recent 7 days, forecast that holdout period, and compare against the real observed values to estimate forecast error.

In [ ]:
daily_df = main_df.groupby('reviewed_date')['review_score'].mean().reset_index()
daily_df.columns = ['ds', 'y']
daily_df['ds'] = pd.to_datetime(daily_df['ds'])
daily_df = daily_df.sort_values('ds').reset_index(drop=True)

cutoff = daily_df['ds'].max() - pd.Timedelta(days=7)
train_df = daily_df[daily_df['ds'] <= cutoff]
test_df = daily_df[daily_df['ds'] > cutoff]

print('train rows:', len(train_df), 'test rows:', len(test_df))

from neuralprophet import NeuralProphet
m = NeuralProphet()
m.fit(train_df, freq='D', learning_rate=0.1)

horizon = len(test_df)
future = m.make_future_dataframe(train_df, periods=horizon)
forecast = m.predict(future)
forecast['yhat1'] = forecast['yhat1'].clip(1, 10)

comparison = test_df.merge(forecast[['ds', 'yhat1']], on='ds', how='left')
comparison['abs_error'] = (comparison['y'] - comparison['yhat1']).abs()

print(comparison[['ds', 'y', 'yhat1', 'abs_error']])
print()

mae = comparison['abs_error'].mean()
rmse = (comparison['abs_error'] ** 2).mean() ** 0.5
print(f"MAE  (avg error, in score points): {mae:.2f}")
print(f"RMSE (penalizes bigger misses more): {rmse:.2f}")